In [1]:
import scanpy as sc
import logging
import rpy2.rinterface_lib.callbacks as rcb
import anndata2ri

rcb.logger.setLevel(logging.ERROR)

%load_ext rpy2.ipython
anndata2ri.set_ipython_converter()


In [2]:
adata = sc.read("/core/cbc/tutorials/workshopdirs/Single-Cell-Transcriptomics/micheli_mouse_muscle/results/adata/05-filter-ambient.h5ad")

In [3]:
adata.obs["sample"].value_counts()

sample
D5_C    5467
D5_B    4754
D7_C    4414
D5_A    4171
D2_C    3824
D0_B    3608
D7_D    3391
D0_A    2287
D2_D    1948
Name: count, dtype: int64

In [4]:
mat = adata.X.T
samples = adata.obs["sample"]

# Detect Doublets

In [5]:
%%R -i mat -i samples -o doublet_score -o doublet_class 

suppressMessages(library(scDblFinder))

set.seed(1234)
sce <- SingleCellExperiment(list(counts=mat))
colData(sce)$samples <- samples
sce <- scDblFinder(sce, samples="samples", clusters=TRUE)
doublet_score <- sce$scDblFinder.score
doublet_class <- sce$scDblFinder.class

  |======================================================================| 100%



In [6]:
adata.obs["scdblfinder_score"] = doublet_score
adata.obs["scdblfinder_class"] = doublet_class

In [7]:
adata.obs["scdblfinder_class"].value_counts()

scdblfinder_class
1    31713
2     2151
Name: count, dtype: int64

In [8]:
adata.write("../results/adata/06-doublets.h5ad")